In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MCC_ETL")
    .master("local[8]")  # use 8 cores to leave some headroom for OS
    .config("spark.driver.memory", "12g")
    .config("spark.executor.memory", "12g")
    .config("spark.sql.shuffle.partitions", "16")  # number of parallel shuffle tasks
    .config("spark.default.parallelism", "16")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")


25/08/27 16:55:45 WARN Utils: Your hostname, Js-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.36.130 instead (on interface en0)
25/08/27 16:55:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/27 16:55:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.6


In [2]:
# File path from notebooks directory
file_path = "../data/raw/mcc_raw.csv"

# Load CSV
df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

# Preview
df.printSchema()
df.show(5, truncate=100)


root
 |-- _c0: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- cik: integer (nullable = true)
 |-- company.name: string (nullable = true)
 |-- form.type: string (nullable = true)
 |-- date.filed: date (nullable = true)
 |-- edgar.link: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- index.link: string (nullable = true)
 |-- contract.link: string (nullable = true)
 |-- exhibit: string (nullable = true)
 |-- description: string (nullable = true)
 |-- exhibit_lead: string (nullable = true)
 |-- contract: string (nullable = true)
 |-- type_label: string (nullable = true)
 |-- type_score: string (nullable = true)
 |-- amend: string (nullable = true)
 |-- restate: string (nullable = true)
 |-- joinder: double (nullable = true)
 |-- termination: integer (nullable = true)
 |-- parties: string (nullable = true)
 |-- agreement_type: string (nullable = true)
 |-- parties_cleaned: string (nullable = true)
 |-- master_parties: string (nullable = true)

+---

25/08/27 16:56:01 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , year, cik, company.name, form.type, date.filed, edgar.link, quarter, index.link, contract.link, exhibit, description, exhibit_lead, contract, type_label, type_score, amend, restate, joinder, termination, parties, agreement_type, parties_cleaned, master_parties
 Schema: _c0, year, cik, company.name, form.type, date.filed, edgar.link, quarter, index.link, contract.link, exhibit, description, exhibit_lead, contract, type_label, type_score, amend, restate, joinder, termination, parties, agreement_type, parties_cleaned, master_parties
Expected: _c0 but found: 
CSV file: file:///Users/jefferyjapheth/Documents/dev/devsuite/project/contract_nlp/data/raw/mcc_raw.csv


In [4]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("company.name", "company_name") \
       .withColumnRenamed("form.type", "form_type") \
       .withColumnRenamed("date.filed", "date_filed")

selected_columns = [
    "year",
    "company_name",
    "form_type",
    "date_filed",
    "description",
    "contract",
    "type_label",
    "type_score",
    "amend",
    "restate",
    "joinder",
    "termination",
    "agreement_type"
]

df = df.select(*selected_columns)
df.show(5, truncate=100)


+----+-----------------------+---------+----------+------------------------+----------------------------------------------------------------------------+----------+------------------+-----+-------+-------+-----------+--------------+
|year|           company_name|form_type|date_filed|             description|                                                                    contract|type_label|        type_score|amend|restate|joinder|termination|agreement_type|
+----+-----------------------+---------+----------+------------------------+----------------------------------------------------------------------------+----------+------------------+-----+-------+-------+-----------+--------------+
|2000|STOCKWALK COM GROUP INC|     10-Q|2000-02-14|asset purchase agreement|/Archives/edgar/data/1001136/000095012400000624/0000950124-00-000624-d2.html|   LABEL_4|0.9999955892562866|    0|      0|    0.0|          0|   purchase&ma|
|2000|STOCKWALK COM GROUP INC|     10-Q|2000-02-14| amendment to emp

In [6]:
df.createOrReplaceTempView("contracts")

spark.sql("""
    SELECT type_label, COUNT(*) AS frequency
    FROM contracts
    WHERE type_label IS NOT NULL
    GROUP BY type_label
    ORDER BY frequency DESC
""").show()



+--------------------+---------+
|          type_label|frequency|
+--------------------+---------+
|             LABEL_1|   486331|
|             LABEL_0|   362605|
|             LABEL_4|   141657|
|             LABEL_3|    89836|
|             LABEL_5|    55881|
|             LABEL_6|    46146|
|             LABEL_2|    36707|
|             LABEL_7|    34898|
|               EX-10|       18|
|/Archives/edgar/d...|        2|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
|               2000"|        1|
|/Archives/edgar/d...|        1|
|/Archives/edgar/d...|        1|
+--------------------+---------+
only showing top 20 rows



In [7]:
# Use Spark SQL to keep only LABEL_0 to LABEL_7
cleaned_df = spark.sql("""
    SELECT *
    FROM contracts
    WHERE type_label IN ('LABEL_0','LABEL_1','LABEL_2','LABEL_3',
                         'LABEL_4','LABEL_5','LABEL_6','LABEL_7')
""")

# Quick check
cleaned_df.groupBy("type_label").count().orderBy("count", ascending=False).show()


+----------+------+
|type_label| count|
+----------+------+
|   LABEL_1|486331|
|   LABEL_0|362605|
|   LABEL_4|141657|
|   LABEL_3| 89836|
|   LABEL_5| 55881|
|   LABEL_6| 46146|
|   LABEL_2| 36707|
|   LABEL_7| 34898|
+----------+------+

